In [1]:
from pathlib import Path
import time
import os
import shutil

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
import numpy as np

RecreerRepertoiresImages = True
IMG_SIZE = (224, 224)

In [3]:
%%time

BaseDataDir = "/data3a/training"

project_root = Path.cwd()
print("project_root : " + str(project_root))

data_dir_src = Path(str(project_root) + BaseDataDir)
print("data_dir_src:", data_dir_src)

project_root : c:\Users\huber_otpq54a\OneDrive\Documents\Formation\IA\Developpement\Projets\Deep_Learning V3
data_dir_src: c:\Users\huber_otpq54a\OneDrive\Documents\Formation\IA\Developpement\Projets\Deep_Learning V3\data3a\training
CPU times: total: 0 ns
Wall time: 0 ns


In [4]:
%%time
# Augmentation réelle du dataset

# CONFIG
INPUT_DIR = data_dir_src
OUTPUT_DIR = Path(str(project_root) + "/CarDamageSeverityDatasetAugmented_train")
AUG_PER_IMAGE = 5  # nombre d'images générées par image

# AUGMENTATION
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.2),
])

if RecreerRepertoiresImages:

    # CRÉER DOSSIERS
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # TRAITEMENT
    for class_name in os.listdir(INPUT_DIR):
        class_input_path = os.path.join(INPUT_DIR, class_name)
        class_output_path = os.path.join(OUTPUT_DIR, class_name)

        os.makedirs(class_output_path, exist_ok=True)

        for img_name in os.listdir(class_input_path):
            img_path = os.path.join(class_input_path, img_name)

            # Charger image
            img = Image.open(img_path).resize(IMG_SIZE)
            img_array = np.array(img)
            img_array = np.expand_dims(img_array, axis=0)  # batch dimension

            # Sauvegarder image originale (optionnel)
            base_name = os.path.splitext(img_name)[0]

            for i in range(AUG_PER_IMAGE):
                augmented = data_augmentation(img_array, training=True)

                # Convertir en image
                aug_img = augmented[0].numpy().astype("uint8")
                aug_img = Image.fromarray(aug_img)

                # Sauvegarde
                new_name = f"{base_name}_aug_{i}.jpg"
                aug_img.save(os.path.join(class_output_path, new_name))

    print("Dataset augmenté créé ✅")


Dataset augmenté créé ✅
CPU times: total: 3min 2s
Wall time: 2min 22s


In [5]:
%%time
# Creation dataset complet : original + augmenté

data_dir_complet = str(project_root) + "/CarDamageSeverityDatasetComplet_train"
data_dir_complet = Path(data_dir_complet)
print("data_dir_complet :", data_dir_complet)

if RecreerRepertoiresImages:
    shutil.copytree(str(data_dir_src), str(data_dir_complet), dirs_exist_ok=True)   
    shutil.copytree(str(OUTPUT_DIR), str(data_dir_complet), dirs_exist_ok=True)   

data_dir_complet : c:\Users\huber_otpq54a\OneDrive\Documents\Formation\IA\Developpement\Projets\Deep_Learning V3\CarDamageSeverityDatasetComplet_train
CPU times: total: 6.33 s
Wall time: 7.75 s
